# News Classifier — BBC Articles

Clasificador de noticias usando Transformers sobre el dataset BBC Articles.

**Luisa Fernanda Aristizabal, Ricardo Jose Garzon y Maria Fernanda Hurtado gomez**

### 1. Instalación de librerías e importacion 

In [1]:
%pip install -q --upgrade pip
%pip install -q kagglehub pandas numpy scikit-learn matplotlib seaborn transformers datasets accelerate nltk
%pip install -q --upgrade --force-reinstall torch --index-url https://download.pytorch.org/whl/cu128

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset as HFDataset

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('Python:', os.sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA no esta disponible en este kernel. '
        'Ejecuta la celda de instalacion, reinicia el kernel y vuelve a correr desde el inicio.'
    )

device = torch.device('cuda:0')
torch.cuda.set_device(device)
print(f'Usando device: {device} - {torch.cuda.get_device_name(0)}')

c:\Users\USUARIO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: c:\Users\USUARIO\AppData\Local\Python\pythoncore-3.14-64\python.exe
PyTorch: 2.11.0+cu128
CUDA build: 12.8
Usando device: cuda:0 - NVIDIA GeForce RTX 5060


In [ ]:
# Descarga al cache global de kagglehub
path = kagglehub.dataset_download("jacopoferretti/bbc-articles-dataset")
print('Cache:', path)

# Copia los CSVs a una carpeta data/ dentro del proyecto
PROJECT_DIR = os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith('.csv'):
            src = os.path.join(root, f)
            dst = os.path.join(DATA_DIR, f)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
            print('->', dst)

Cache: /Users/mariafernandahurtadogomez/.cache/kagglehub/datasets/jacopoferretti/bbc-articles-dataset/versions/16
-> /Users/mariafernandahurtadogomez/Desktop/transformers/DL-TRANSFORMERS/data/bbc_news_text_complexity_summarization.csv
-> /Users/mariafernandahurtadogomez/Desktop/transformers/DL-TRANSFORMERS/data/bbc-news-data.csv
-> /Users/mariafernandahurtadogomez/Desktop/transformers/DL-TRANSFORMERS/data/bbc_text_cls.csv


In [8]:
# Carga desde la carpeta data/ del proyecto
csv_files = sorted([os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith('.csv')])
print('CSVs en data/:', csv_files)

# Usamos bbc_text_cls.csv que es el más limpio para clasificación (text + labels)
df = pd.read_csv(os.path.join(DATA_DIR, 'bbc_text_cls.csv'))
df.head()

CSVs en data/: ['/Users/mariafernandahurtadogomez/Desktop/transformers/DL-TRANSFORMERS/data/bbc-news-data.csv', '/Users/mariafernandahurtadogomez/Desktop/transformers/DL-TRANSFORMERS/data/bbc_news_text_complexity_summarization.csv', '/Users/mariafernandahurtadogomez/Desktop/transformers/DL-TRANSFORMERS/data/bbc_text_cls.csv']


,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


## 4. Preprocesamiento

Como vamos a usar Transformers (BERT/DistilBERT), la limpieza es **mínima**: el tokenizer del modelo se encarga del resto. NO hacemos lowercasing, NO removemos stopwords, NO lematizamos.

### 4.1 EDA básico

In [ ]:
print('Shape:', df.shape)
print('\nColumnas:', df.columns.tolist())
print('\nNulos por columna:')
print(df.isnull().sum())
print('\nDuplicados:', df.duplicated().sum())
print('\nDistribución de clases:')
print(df['labels'].value_counts())

In [ ]:
# Distribución visual de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['labels'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribución de clases')
axes[0].set_xlabel('Categoría')
axes[0].set_ylabel('Cantidad de artículos')
axes[0].tick_params(axis='x', rotation=45)

# Distribución de longitudes (en palabras) para definir max_length del tokenizer
df['n_words'] = df['text'].str.split().str.len()
axes[1].hist(df['n_words'], bins=50, color='coral', edgecolor='black')
axes[1].set_title('Distribución de longitudes (palabras)')
axes[1].set_xlabel('Número de palabras')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(df['n_words'].median(), color='black', linestyle='--', label=f"mediana={df['n_words'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Longitud (palabras) — min: {df['n_words'].min()}, max: {df['n_words'].max()}, "
      f"media: {df['n_words'].mean():.0f}, p95: {df['n_words'].quantile(0.95):.0f}")

### 4.2 Limpieza mínima

Solo lo necesario: drop nulos/duplicados, normalizar whitespace y remover URLs/HTML si los hay. **Mantenemos** mayúsculas, puntuación y stopwords.

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r'http\S+|www\.\S+', ' ', text)   # URLs
    text = re.sub(r'<.*?>', ' ', text)              # HTML
    text = re.sub(r'\s+', ' ', text)                # whitespace múltiple
    return text.strip()

# Aplicar limpieza
df = df.dropna(subset=['text', 'labels']).copy()
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
df['text'] = df['text'].astype(str).apply(clean_text)

# Eliminar textos vacíos tras limpieza
df = df[df['text'].str.len() > 0].reset_index(drop=True)

print('Shape después de limpieza:', df.shape)
df.head(2)

### 4.3 Encoding de labels

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['labels'])

# Mapeos id <-> nombre (los necesita el modelo)
id2label = {i: c for i, c in enumerate(le.classes_)}
label2id = {c: i for i, c in enumerate(le.classes_)}

print('Clases:', list(le.classes_))
print('id2label:', id2label)
print('Num labels:', len(le.classes_))

### 4.4 Split estratificado: train / val / test (70/15/15)

In [ ]:
SEED = 42

# Primero separamos test (15%)
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED
)

# Luego separamos val del resto (15% del total ≈ 17.6% del 85% restante)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.15/0.85, stratify=train_val_df['label'], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\nDistribución por split:')
print(pd.DataFrame({
    'train': train_df['labels'].value_counts(normalize=True).round(3),
    'val':   val_df['labels'].value_counts(normalize=True).round(3),
    'test':  test_df['labels'].value_counts(normalize=True).round(3),
}))

## 5. Tokenización con el modelo

Cargamos el tokenizer del modelo elegido (RoBERTa) y tokenizamos los 3 splits. La limpieza ya está hecha en la sección 4 — aquí solo convertimos texto a IDs.

### 5.1 Configuración: modelo y `max_length`

`max_length` lo elegimos a partir de la distribución de longitudes (sección 4.1). Como referencia rápida:
- p95 ≤ 200 tokens → `max_length=256`
- p95 ≤ 400 tokens → `max_length=512` (límite de BERT/DistilBERT)

Truncamos lo que pase del límite. Padding NO se hace acá — lo aplica el `DataCollatorWithPadding` dinámicamente por batch (más eficiente).

In [ ]:
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 256  # ajusta según el p95 que viste en 4.1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer cargado: {MODEL_NAME}')
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Max model length: {tokenizer.model_max_length}')
print(f'Special tokens: {tokenizer.all_special_tokens}')

### 5.2 Verificación: longitudes reales en tokens

Antes de fijar `MAX_LENGTH`, miramos la distribución de longitudes en **tokens** (no palabras), que es lo que realmente le entra al modelo. Si el p95 en tokens supera mucho `MAX_LENGTH`, súbelo o asume que vas a truncar.

In [ ]:
sample_lengths = [len(tokenizer.encode(t, add_special_tokens=True, truncation=False))
                  for t in train_df['text'].sample(min(500, len(train_df)), random_state=SEED)]

import numpy as np
print(f'Longitudes en tokens (muestra de {len(sample_lengths)}):')
print(f'  min: {min(sample_lengths)}')
print(f'  media: {np.mean(sample_lengths):.0f}')
print(f'  mediana: {np.median(sample_lengths):.0f}')
print(f'  p95: {np.percentile(sample_lengths, 95):.0f}')
print(f'  max: {max(sample_lengths)}')
print(f'\n% de textos que se truncarán con MAX_LENGTH={MAX_LENGTH}: '
      f'{100 * np.mean(np.array(sample_lengths) > MAX_LENGTH):.1f}%')

### 5.3 Dataset de PyTorch

Creamos una clase `NewsDataset` que hereda de `torch.utils.data.Dataset`. Solo necesita 3 métodos:

- `__init__`: guarda los textos, las etiquetas y el tokenizer.
- `__len__`: cuántos ejemplos hay.
- `__getitem__(idx)`: tokeniza **un** ejemplo y devuelve un dict con `input_ids`, `attention_mask` y `labels` ya como tensores.

Esto es lo mismo que verías en cualquier curso de PyTorch — nada de HuggingFace `datasets` ni collators.

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',     # rellena hasta max_length (todos los ejemplos misma forma)
            max_length=self.max_length,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),       # (max_length,)
            'attention_mask': enc['attention_mask'].squeeze(0),  # (max_length,)
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = NewsDataset(train_df['text'], train_df['label'], tokenizer, MAX_LENGTH)
val_dataset   = NewsDataset(val_df['text'],   val_df['label'],   tokenizer, MAX_LENGTH)
test_dataset  = NewsDataset(test_df['text'],  test_df['label'],  tokenizer, MAX_LENGTH)

print(f'train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}')
print('\nEjemplo train_dataset[0]:')
sample = train_dataset[0]
for k, v in sample.items():
    print(f'  {k}: shape={tuple(v.shape)}, dtype={v.dtype}')

 DataLoaders

`DataLoader` agrupa los ejemplos en batches y los baraja. Con esto ya tienes lo que necesitas para entrenar.

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Verificar un batch
batch = next(iter(train_loader))
print('Keys del batch:', list(batch.keys()))
print('input_ids shape:', batch['input_ids'].shape)        # (BATCH_SIZE, MAX_LENGTH)
print('attention_mask shape:', batch['attention_mask'].shape)
print('labels shape:', batch['labels'].shape)
print('labels:', batch['labels'])